# KΛIROS — MusicGen Audio Generator

**Before running:** Make sure you are using a GPU runtime.
> Runtime → Change runtime type → T4 GPU → Save

Workflow:
1. Run Cell 1 — installs audiocraft (takes ~2 min first time)
2. Run Cell 2 — loads the model (downloads ~1.5GB first time)
3. Paste your prompt in Cell 3 and run it
4. Download the generated WAV from Cell 4

In [ ]:
# ── CELL 1: Install dependencies ─────────────────────────────────────────────
!pip install -q audiocraft
print('Done. Restart runtime if prompted, then continue from Cell 2.')

In [ ]:
# ── CELL 2: Load model ────────────────────────────────────────────────────────
import torch
from audiocraft.models import MusicGen

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — switch to GPU runtime!"}')

# medium = best quality/speed balance. Use 'large' for higher quality (needs more VRAM)
model = MusicGen.get_pretrained('facebook/musicgen-medium')
print('Model loaded.')

In [ ]:
# ── CELL 3: Generate audio ────────────────────────────────────────────────────
# Paste your prompt from export_prompts.py output below

PROMPT = """A 124 BPM melodic techno track in F minor. Ethereal, ritual, and liminal atmosphere. \
Features warm sub-bass pulse, subtle acid line weaving through the mix, rolling four-on-the-floor kick, \
and deep melodic bassline. Inspired by Monolink and ARTBAT. Slow build over 8 bars. \
No lyrics except one whispered phrase buried deep in reverb. Suitable for a Berlin underground \
after-hours set. Sacred geometry energy — mathematical, meditative, hypnotic."""

TITLE    = "Stasis"       # change to your track title from export_prompts
DROP     = "morning"      # morning or evening
DURATION = 30             # seconds — increase for longer previews (max ~60 recommended)

# ─────────────────────────────────────────────────────────────────────────────
model.set_generation_params(duration=DURATION)

print(f'Generating: "{TITLE}" [{DROP} drop] — {DURATION}s ...')
with torch.inference_mode():
    wav = model.generate([PROMPT])

print('Done.')

# Preview in notebook
from audiocraft.utils.notebook import display_audio
display_audio(wav, sample_rate=model.sample_rate)

In [ ]:
# ── CELL 4: Save and download ─────────────────────────────────────────────────
import torchaudio
import datetime
from google.colab import files

date_str  = datetime.date.today().strftime('%Y%m%d')
safe_title = TITLE.lower().replace(' ', '_')
filename  = f"transmission_{date_str}_{DROP}_{safe_title}.wav"

torchaudio.save(filename, wav[0].cpu(), model.sample_rate)
print(f'Saved: {filename}')

files.download(filename)
print('Download started.')

In [ ]:
# ── CELL 5 (optional): Generate BOTH drops back to back ──────────────────────
# Paste both prompts from prompts_today.json here

TRACKS = [
    {
        "drop": "morning",
        "title": "Stasis",
        "prompt": """A 124 BPM melodic techno track in F minor. Ethereal, ritual, and liminal atmosphere. \
Features warm sub-bass pulse, subtle acid line weaving through the mix, rolling four-on-the-floor kick, \
and deep melodic bassline. Inspired by Monolink and ARTBAT. Slow build over 8 bars. \
No lyrics except one whispered phrase buried deep in reverb. Sacred geometry energy.""",
    },
    {
        "drop": "evening",
        "title": "Torque Signal",
        "prompt": """A 134 BPM peak-time industrial techno track in A minor. Machine-like, unforgiving, \
and frenetic energy. Features dark atmospheric drone, metallic percussion hits, filtered white noise sweeps, \
and punishing kick drum with distorted clap. Inspired by Alignment and Surgeon. Hard, pounding, relentless. \
Built for peak-time warehouse raving. No breakdowns — pure momentum.""",
    },
]

model.set_generation_params(duration=30)
date_str = datetime.date.today().strftime('%Y%m%d')

for t in TRACKS:
    print(f"Generating {t['drop']} drop: '{t['title']}' ...")
    with torch.inference_mode():
        wav = model.generate([t['prompt']])
    safe = t['title'].lower().replace(' ', '_')
    fname = f"transmission_{date_str}_{t['drop']}_{safe}.wav"
    torchaudio.save(fname, wav[0].cpu(), model.sample_rate)
    files.download(fname)
    print(f'  Downloaded: {fname}')

print('Both tracks done!')